# 集群上lumpy运行流程

## 一、数据来源

### 197个样本路径：/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams/

### 80组配对样本名称列表：/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt

## 二、配置环境与运行 

In [ ]:
conda create -n gridss_env r-base=4.3.2 -y
conda activate gridss_env

conda install -c bioconda -c conda-forge gridss hmftools-gripss bcftools samtools -y

In [ ]:
# 找安装在conda里面的 jar包
find $CONDA_PREFIX -name "gripss.jar"
# 找到路径为：/mnt/home/ygjx/chenkejin/anaconda3/envs/gridss_env/share/hmftools-gripss-2.4-0/gripss.jar

In [ ]:
# 看GRIPSS的版本及参数，显示版本为2.4
gripss -help

## 三、多样本批量处理

In [ ]:
# 上传作业
sbatch batch_gridss.sh

### batch_gridss.sh代码如下：

In [ ]:
#!/bin/bash
#SBATCH --job-name=GRIDSS_GRIPSS_Batch
#SBATCH --nodes=1
#SBATCH --cpus-per-task=16
#SBATCH --mem=64G
#SBATCH --array=1-80%20
#SBATCH --output=/mnt/home/ygjx/chenkejin/GRIDSS/logs/slurm_array_%A_%a.out 

set -euo pipefail

# --- 0. 激活正确的实际环境 ---
source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate gridss_env

# --- 1. 全局变量与路径 ---
REF_FA="/mnt/home/ygjx/chenkejin/delly/delly-main/Homo_sapiens_assembly38.fasta"
BLACKLIST="/mnt/home/ygjx/chenkejin/GRIDSS/tracks/ENCFF356LFX.bed"
THREADS=16
GRIPSS_JAR="/mnt/home/ygjx/chenkejin/anaconda3/envs/gridss_env/share/hmftools-gripss-2.4-0/gripss.jar"

SOURCE_BASE="/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams"
WORK_DIR="/mnt/home/ygjx/chenkejin/GRIDSS"
TASK_LIST="/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt"

MASTER_LOG="${WORK_DIR}/master_progress_gripss.log"

# --- 2. 任务行解析 (无敌防爆版) ---
LINE=$(sed -n "${SLURM_ARRAY_TASK_ID}p" "$TASK_LIST" | tr -d '\r' | xargs)

if [ -z "$LINE" ]; then
    exit 0
fi

NORMAL_ID=$(echo "$LINE" | awk '{print $1}')
TUMOR_ID=$(echo "$LINE" | awk '{print $2}')

if [ -z "$NORMAL_ID" ] || [ -z "$TUMOR_ID" ]; then
    exit 0
fi

if [[ "${NORMAL_ID,,}" == *"normal"* ]] || [[ "${NORMAL_ID,,}" == *"id"* ]]; then
    exit 0
fi

PREFIX=$(echo "$NORMAL_ID" | sed 's/N$//')
if [ -z "$PREFIX" ]; then
    exit 0
fi

# --- 3. 独立样本日志重定向 ---
mkdir -p "${WORK_DIR}/logs"
SAMPLE_LOG="${WORK_DIR}/logs/${PREFIX}.log"
exec > >(tee -i "$SAMPLE_LOG") 2>&1

echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] 样本 ${PREFIX} 开始处理"
echo "全新引擎: GRIDSS + GRIPSS | 16 CPUs, 64G Mem"
echo "任务阵列 ID: ${SLURM_ARRAY_TASK_ID}/80  执行节点: $(hostname)"
echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX}" >> "$MASTER_LOG"

# --- 4. 目录构建与断点续传检测 ---
FINAL_VCF_DIR="${WORK_DIR}/Final_Results"
mkdir -p "$FINAL_VCF_DIR"

RUN_DIR="${WORK_DIR}/sandbox/${PREFIX}_gridss_run"
raw_vcf="${RUN_DIR}/${PREFIX}.gridss.raw.vcf"

somatic_vcf="${FINAL_VCF_DIR}/${TUMOR_ID}.gripss.vcf.gz"
final_unzipped_vcf="${FINAL_VCF_DIR}/${TUMOR_ID}.gripss.vcf"

if [ -f "$final_unzipped_vcf" ]; then
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SKIP] 样本 ${PREFIX} 已存在最终解压结果，跳过。"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SKIP] Sample ${PREFIX} skipped" >> "$MASTER_LOG"
    exit 0
fi

rm -rf "$RUN_DIR" && mkdir -p "$RUN_DIR"
cd "$RUN_DIR"

NORMAL_BAM="${SOURCE_BASE}/${NORMAL_ID}/${NORMAL_ID}.sorted.markdup.BQSR.bam"
TUMOR_BAM="${SOURCE_BASE}/${TUMOR_ID}/${TUMOR_ID}.sorted.markdup.BQSR.bam"

# --- 5. 核心流程执行 ---
{
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 1/3: 运行 GRIDSS Joint Calling..."
    gridss \
        -r "$REF_FA" \
        -o "$raw_vcf" \
        -b "$BLACKLIST" \
        -t "$THREADS" \
        --workingdir "$RUN_DIR" \
        "$NORMAL_BAM" \
        "$TUMOR_BAM"

    echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 2/3: 运行 Java 版 GRIPSS 过滤..."
    if [ -f "$raw_vcf" ]; then
        java -Xmx32G -jar "$GRIPSS_JAR" \
            -sample "$TUMOR_ID" \
            -reference "$NORMAL_ID" \
            -ref_genome_version 38 \
            -ref_genome "$REF_FA" \
            -vcf "$raw_vcf" \
            -output_dir "$FINAL_VCF_DIR" || { echo "⚠️ 致命错误：GRIPSS 运行失败！"; exit 1; }

        echo "[$(date '+%Y-%m-%d %H:%M:%S')] 步骤 3/3: 处理最终输出..."
        if [ -f "$somatic_vcf" ]; then
            gunzip -c "$somatic_vcf" > "$final_unzipped_vcf"
            echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] 样本 ${PREFIX} 全流程处理并成功解压 VCF！"
        else
            echo "⚠️ 提示：样本 ${PREFIX} 经 GRIPSS 质控后无高置信度体细胞突变，创建空文件占位。"
            touch "$final_unzipped_vcf"
            echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] 样本 ${PREFIX} 流程闭环 (0 Variants)。"
        fi
        
        echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX} completed" >> "$MASTER_LOG"
        
        cd "${WORK_DIR}"
        rm -rf "$RUN_DIR"
    else
        echo "⚠️ 错误：未找到 $raw_vcf，GRIDSS 第一阶段运行失败。"
        false
    fi

} || {
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] 样本 ${PREFIX} 运行失败！请检查日志。"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] Task ${SLURM_ARRAY_TASK_ID}/80: Sample ${PREFIX} failed!" >> "$MASTER_LOG"
    exit 1
}


### tail -f master_progress_gripss.log 查看日志

### 结果路径：/mnt/home/ygjx/chenkejin/GRIDSS/Final_Results/

## 四、PON过滤

### 所用PON来源：https://resources.hartwigmedicalfoundation.nl/?utm_source=chatgpt.com

### filter_gridss_gripss_with_pon.sh脚本：

In [ ]:
#!/bin/bash
#SBATCH --job-name=GRIDSS_PON_Filter
#SBATCH --nodes=1
#SBATCH --cpus-per-task=2
#SBATCH --mem=8G
#SBATCH --output=/mnt/home/ygjx/chenkejin/GRIDSS/logs/pon_filter_%A_%a.out
#SBATCH --error=/mnt/home/ygjx/chenkejin/GRIDSS/logs/pon_filter_%A_%a.err

set -euo pipefail

source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate gridss_env

WORK_DIR="${WORK_DIR:-/mnt/home/ygjx/chenkejin/GRIDSS}"
TASK_LIST="${TASK_LIST:-/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt}"
INPUT_DIR="${INPUT_DIR:-${WORK_DIR}/Final_Results}"
OUT_DIR="${OUT_DIR:-${WORK_DIR}/Final_Results_PON_filtered}"
PON_DIR="${PON_DIR:-${WORK_DIR}/PON}"

PON_BREAKPOINT="${PON_BREAKPOINT:-${PON_DIR}/gridss_pon_breakpoint.bedpe}"
PON_SINGLE_BREAKEND="${PON_SINGLE_BREAKEND:-${PON_DIR}/gridss_pon_single_breakend.bed}"

FORCE="${FORCE:-0}"

usage() {
  cat <<'EOF'
Usage:
  Slurm array:
    sbatch --array=1-N%20 filter_gridss_gripss_with_pon.sh

  Single sample pair:
    bash filter_gridss_gripss_with_pon.sh --normal NORMAL_ID --tumor TUMOR_ID

  Summarize after array is complete:
    bash filter_gridss_gripss_with_pon.sh --summarize

Defaults:
  WORK_DIR=/mnt/home/ygjx/chenkejin/GRIDSS
  TASK_LIST=/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt
  INPUT_DIR=${WORK_DIR}/Final_Results
  OUT_DIR=${WORK_DIR}/Final_Results_PON_filtered
  PON_BREAKPOINT=${WORK_DIR}/PON/gridss_pon_breakpoint.bedpe
  PON_SINGLE_BREAKEND=${WORK_DIR}/PON/gridss_pon_single_breakend.bed

Input VCF priority:
  ${INPUT_DIR}/${TUMOR_ID}.gripss.vcf
  ${INPUT_DIR}/${TUMOR_ID}.gripss.vcf.gz

Output:
  ${OUT_DIR}/${TUMOR_ID}.gripss.pon_filtered.vcf
  ${OUT_DIR}/removed/${TUMOR_ID}.gripss.pon_removed.vcf
  ${OUT_DIR}/summary_parts/${PREFIX}.gridss_pon_filter.tsv
  ${OUT_DIR}/all_samples.gridss_pon_filter.summary.tsv
EOF
}

MODE="array"
NORMAL_ID=""
TUMOR_ID=""

while [ "$#" -gt 0 ]; do
  case "$1" in
    --normal)
      NORMAL_ID="${2:?ERROR: --normal needs a sample id}"
      MODE="single"
      shift 2
      ;;
    --tumor)
      TUMOR_ID="${2:?ERROR: --tumor needs a sample id}"
      MODE="single"
      shift 2
      ;;
    --summarize)
      MODE="summarize"
      shift
      ;;
    --force)
      FORCE=1
      shift
      ;;
    -h|--help)
      usage
      exit 0
      ;;
    *)
      echo "ERROR: unknown argument: $1" >&2
      usage >&2
      exit 1
      ;;
  esac
done

mkdir -p "${OUT_DIR}" "${OUT_DIR}/removed" "${OUT_DIR}/summary_parts" "${OUT_DIR}/logs" "${WORK_DIR}/logs"

if [ "${MODE}" = "summarize" ]; then
  SUMMARY="${OUT_DIR}/all_samples.gridss_pon_filter.summary.tsv"
  HEADER="sample\tnormal_id\ttumor_id\tinput_vcf\toutput_vcf\tremoved_vcf\tstatus\tinput_records\toutput_records\tremoved_records\tremoved_breakpoint_pon\tremoved_single_breakend_pon\tremoved_both\tremoved_mate_records\tpon_breakpoint\tpon_single_breakend"
  echo -e "${HEADER}" > "${SUMMARY}"
  find "${OUT_DIR}/summary_parts" -type f -name "*.gridss_pon_filter.tsv" | sort | xargs -r cat >> "${SUMMARY}"

  echo "===== GRIDSS/GRIPSS PON filter summary ====="
  echo -n "Total summarized sample pairs: "
  tail -n +2 "${SUMMARY}" | wc -l
  echo
  echo "Status counts:"
  awk -F'\t' 'NR>1{count[$7]++} END{for (s in count) print s, count[s]}' "${SUMMARY}" | sort
  echo
  echo "Total records before/after/removed:"
  awk -F'\t' 'NR>1{before+=$8; after+=$9; removed+=$10} END{print "before=" before "\nafter=" after "\nremoved=" removed}' "${SUMMARY}"
  echo
  echo "Summary file:"
  echo "${SUMMARY}"
  exit 0
fi

if [ ! -f "${PON_BREAKPOINT}" ]; then
  echo "ERROR: PON breakpoint BEDPE not found: ${PON_BREAKPOINT}" >&2
  exit 1
fi

if [ ! -f "${PON_SINGLE_BREAKEND}" ]; then
  echo "ERROR: PON single breakend BED not found: ${PON_SINGLE_BREAKEND}" >&2
  exit 1
fi

if [ "${MODE}" = "array" ]; then
  if [ -z "${SLURM_ARRAY_TASK_ID:-}" ]; then
    echo "ERROR: no --normal/--tumor provided and SLURM_ARRAY_TASK_ID is not set." >&2
    usage >&2
    exit 1
  fi

  if [ ! -f "${TASK_LIST}" ]; then
    echo "ERROR: task list not found: ${TASK_LIST}" >&2
    exit 1
  fi

  LINE="$(awk -v idx="${SLURM_ARRAY_TASK_ID}" '
    NF>=2 && $1 !~ /^#/ && tolower($1) !~ /normal|sample|id/ {
      n++
      if (n==idx) {
        print $1 "\t" $2
        exit
      }
    }
  ' "${TASK_LIST}" | tr -d '\r')"

  if [ -z "${LINE}" ]; then
    echo "ERROR: empty task line for SLURM_ARRAY_TASK_ID=${SLURM_ARRAY_TASK_ID}" >&2
    exit 1
  fi

  IFS=$'\t' read -r NORMAL_ID TUMOR_ID <<< "${LINE}"
fi

if [ -z "${NORMAL_ID}" ] || [ -z "${TUMOR_ID}" ]; then
  echo "ERROR: NORMAL_ID or TUMOR_ID is empty." >&2
  exit 1
fi

PREFIX="${NORMAL_ID%N}"
LOG="${OUT_DIR}/logs/${PREFIX}.gridss_pon_filter.log"
exec > >(tee -i "${LOG}") 2>&1

INPUT_VCF=""
if [ -f "${INPUT_DIR}/${TUMOR_ID}.gripss.vcf" ]; then
  INPUT_VCF="${INPUT_DIR}/${TUMOR_ID}.gripss.vcf"
elif [ -f "${INPUT_DIR}/${TUMOR_ID}.gripss.vcf.gz" ]; then
  INPUT_VCF="${INPUT_DIR}/${TUMOR_ID}.gripss.vcf.gz"
else
  echo "ERROR: input GRIPSS VCF not found for ${TUMOR_ID}" >&2
  echo "Checked:" >&2
  echo "  ${INPUT_DIR}/${TUMOR_ID}.gripss.vcf" >&2
  echo "  ${INPUT_DIR}/${TUMOR_ID}.gripss.vcf.gz" >&2
  exit 1
fi

OUTPUT_VCF="${OUT_DIR}/${TUMOR_ID}.gripss.pon_filtered.vcf"
REMOVED_VCF="${OUT_DIR}/removed/${TUMOR_ID}.gripss.pon_removed.vcf"
SUMMARY_PART="${OUT_DIR}/summary_parts/${PREFIX}.gridss_pon_filter.tsv"

if [ "${FORCE}" != "1" ] && [ -f "${OUTPUT_VCF}" ] && [ -f "${SUMMARY_PART}" ]; then
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SKIP] Output already exists for ${PREFIX}. Use --force or FORCE=1 to rerun."
  exit 0
fi

echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] GRIDSS/GRIPSS PON filtering: ${PREFIX}"
echo "Normal: ${NORMAL_ID}"
echo "Tumor : ${TUMOR_ID}"
echo "Input : ${INPUT_VCF}"
echo "Output: ${OUTPUT_VCF}"
echo "Removed records: ${REMOVED_VCF}"
echo "PON breakpoint    : ${PON_BREAKPOINT}"
echo "PON single breakend: ${PON_SINGLE_BREAKEND}"
echo "=========================================================="

python - "${INPUT_VCF}" "${OUTPUT_VCF}" "${REMOVED_VCF}" "${SUMMARY_PART}" "${PREFIX}" "${NORMAL_ID}" "${TUMOR_ID}" "${PON_BREAKPOINT}" "${PON_SINGLE_BREAKEND}" <<'PY'
import gzip
import re
import sys
from collections import defaultdict

(
    input_vcf,
    output_vcf,
    removed_vcf,
    summary_part,
    sample,
    normal_id,
    tumor_id,
    pon_breakpoint,
    pon_single_breakend,
) = sys.argv[1:]

BIN_SIZE = 1_000_000


def open_text(path, mode="rt"):
    if path.endswith(".gz"):
        return gzip.open(path, mode)
    return open(path, mode)


def bin_range(start, end):
    if end <= start:
        end = start + 1
    return range(start // BIN_SIZE, (end - 1) // BIN_SIZE + 1)


def load_bed(path):
    bins = defaultdict(list)
    n = 0
    with open(path) as handle:
        for line in handle:
            if not line.strip() or line.startswith("#"):
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 3:
                continue
            chrom = fields[0]
            try:
                start = int(fields[1])
                end = int(fields[2])
            except ValueError:
                continue
            if end <= start:
                end = start + 1
            for b in bin_range(start, end):
                bins[(chrom, b)].append((start, end))
            n += 1
    return bins, n


def load_bedpe(path):
    bins = defaultdict(list)
    n = 0
    with open(path) as handle:
        for line in handle:
            if not line.strip() or line.startswith("#"):
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 6:
                continue
            c1, c2 = fields[0], fields[3]
            try:
                s1, e1 = int(fields[1]), int(fields[2])
                s2, e2 = int(fields[4]), int(fields[5])
            except ValueError:
                continue
            if e1 <= s1:
                e1 = s1 + 1
            if e2 <= s2:
                e2 = s2 + 1

            rec = (s1, e1, s2, e2)
            for b1 in bin_range(s1, e1):
                for b2 in bin_range(s2, e2):
                    bins[(c1, c2, b1, b2)].append(rec)

            rec_rev = (s2, e2, s1, e1)
            for b2 in bin_range(s2, e2):
                for b1 in bin_range(s1, e1):
                    bins[(c2, c1, b2, b1)].append(rec_rev)
            n += 1
    return bins, n


def point_hits_bed(chrom, pos0, bins):
    for start, end in bins.get((chrom, pos0 // BIN_SIZE), []):
        if start <= pos0 < end:
            return True
    return False


def breakpoint_hits_bedpe(c1, pos1_0, c2, pos2_0, bins):
    for s1, e1, s2, e2 in bins.get((c1, c2, pos1_0 // BIN_SIZE, pos2_0 // BIN_SIZE), []):
        if s1 <= pos1_0 < e1 and s2 <= pos2_0 < e2:
            return True
    return False


def parse_info(info):
    out = {}
    for item in info.split(";"):
        if not item:
            continue
        if "=" in item:
            key, value = item.split("=", 1)
            out[key] = value
        else:
            out[item] = True
    return out


def parse_alt_mate(alt):
    match = re.search(r"[\[\]]([^:\[\]]+):([0-9]+)[\[\]]", alt)
    if match:
        return match.group(1), int(match.group(2))
    return None, None


def add_filter_reason(line, reason):
    fields = line.rstrip("\n").split("\t")
    if len(fields) < 8:
        return line
    if fields[7] in (".", ""):
        fields[7] = "PON_FILTER_REASON=" + reason
    else:
        fields[7] += ";PON_FILTER_REASON=" + reason
    return "\t".join(fields) + "\n"


sgl_bins, sgl_n = load_bed(pon_single_breakend)
bp_bins, bp_n = load_bedpe(pon_breakpoint)

headers = []
records = []

with open_text(input_vcf, "rt") as handle:
    for line in handle:
        if line.startswith("#"):
            headers.append(line)
            continue
        if not line.strip():
            continue
        fields = line.rstrip("\n").split("\t")
        if len(fields) < 8:
            continue
        records.append(fields + [line])

remove_ids = set()
direct_reason_by_id = {}
direct_remove_lines = {}

input_records = len(records)
removed_breakpoint = 0
removed_single = 0
removed_both = 0

for fields in records:
    chrom = fields[0]
    try:
        pos = int(fields[1])
    except ValueError:
        continue
    pos0 = pos - 1
    vid = fields[2]
    alt = fields[4]
    info = parse_info(fields[7])

    mate_chrom, mate_pos = parse_alt_mate(alt)
    if mate_chrom is None and "CHR2" in info and "END" in info:
        try:
            mate_chrom = str(info["CHR2"])
            mate_pos = int(str(info["END"]).split(",")[0])
        except ValueError:
            mate_chrom, mate_pos = None, None
    elif mate_chrom is None and "END" in info:
        try:
            mate_chrom = chrom
            mate_pos = int(str(info["END"]).split(",")[0])
        except ValueError:
            mate_chrom, mate_pos = None, None

    hit_bp = False
    if mate_chrom is not None and mate_pos is not None:
        hit_bp = breakpoint_hits_bedpe(chrom, pos0, mate_chrom, mate_pos - 1, bp_bins)

    hit_sgl = point_hits_bed(chrom, pos0, sgl_bins)

    if not hit_bp and not hit_sgl:
        continue

    if hit_bp and hit_sgl:
        reason = "PON_BREAKPOINT_AND_SINGLE_BREAKEND"
        removed_both += 1
    elif hit_bp:
        reason = "PON_BREAKPOINT"
        removed_breakpoint += 1
    else:
        reason = "PON_SINGLE_BREAKEND"
        removed_single += 1

    key = vid if vid not in ("", ".") else f"{chrom}:{pos}:{alt}"
    remove_ids.add(key)
    direct_reason_by_id[key] = reason
    direct_remove_lines[key] = add_filter_reason(fields[-1], reason)

    mate_ids = str(info.get("MATEID", "")).split(",")
    for mate_id in mate_ids:
        if mate_id and mate_id != ".":
            remove_ids.add(mate_id)
            direct_reason_by_id.setdefault(mate_id, "MATE_OF_PON_HIT")

output_count = 0
removed_count = 0
removed_mate_records = 0

with open(output_vcf, "wt") as out, open(removed_vcf, "wt") as rem:
    for h in headers:
        out.write(h)
        rem.write(h)

    for fields in records:
        chrom = fields[0]
        pos = fields[1]
        vid = fields[2]
        alt = fields[4]
        key = vid if vid not in ("", ".") else f"{chrom}:{pos}:{alt}"
        line = fields[-1]
        if key in remove_ids:
            reason = direct_reason_by_id.get(key, "PON_HIT")
            if reason == "MATE_OF_PON_HIT":
                removed_mate_records += 1
            rem.write(add_filter_reason(line, reason))
            removed_count += 1
        else:
            out.write(line)
            output_count += 1

status = "PASS"

with open(summary_part, "wt") as summary:
    summary.write(
        "\t".join(
            [
                sample,
                normal_id,
                tumor_id,
                input_vcf,
                output_vcf,
                removed_vcf,
                status,
                str(input_records),
                str(output_count),
                str(removed_count),
                str(removed_breakpoint),
                str(removed_single),
                str(removed_both),
                str(removed_mate_records),
                pon_breakpoint,
                pon_single_breakend,
            ]
        )
        + "\n"
    )

print("PON breakpoint intervals:", bp_n)
print("PON single-breakend intervals:", sgl_n)
print("Input records:", input_records)
print("Output records:", output_count)
print("Removed records:", removed_count)
print("Removed direct breakpoint hits:", removed_breakpoint)
print("Removed direct single-breakend hits:", removed_single)
print("Removed direct both hits:", removed_both)
print("Removed mate records:", removed_mate_records)
PY

echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] ${PREFIX}"
echo "Filtered VCF: ${OUTPUT_VCF}"
echo "Removed VCF : ${REMOVED_VCF}"
echo "Summary part: ${SUMMARY_PART}"


### 批量提交：

In [ ]:
TASK_LIST="/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt"
N=$(awk 'NF>=2 && $1 !~ /^#/{n++} END{print n+0}' "${TASK_LIST}")

sbatch --array=1-${N}%40 filter_gridss_gripss_with_pon.sh

### 过滤完成后汇总：

In [ ]:
bash filter_gridss_gripss_with_pon.sh --summarize